# Chapter 2 standalone: the engineering mindset, enforced

*Google Gemini edition*

**Author: Imran Ahmad** · Companion notebook for *Building Reliable AI-Assisted Software Systems* (Packt). All characters, companies, incidents, and data are fictional.

## Objectives

Chapter 2 replaces the artisanal habit (tune by feel, ship on vibes) with a systematic one: quality lives in a process, and machinery enforces it. This notebook reproduces the chapter's enforcement beats offline:

- five runs of one refund prompt, one of which promises $4,200 the policy does not allow
- an output guard that blocks that promise against the $400 approval ceiling
- an input guard that stops a prompt injection before it reaches the model
- redaction on inbound personal data
- the honest arithmetic of unbounded automation: 13 retries at $30/hr for 60 hours

Everything runs from a seeded support layer: no key, no network, byte-identical numbers on every run.

<img src="assets/fig_2_4_request_journey_color_300dpi.png" width="700" alt="request journey"/>

One support ticket travels through every pillar above; the cells below build the checkpoints it passes.

### Provider edition: Google Gemini

This edition adds one optional live cell that sends the refund prompt to a real model and runs the chapter's output guard on the reply. Requirements beyond the base bundle: `google-genai` (see `requirements-providers.txt`). Set `GEMINI_API_KEY` to enable the live probe. **Simulation Mode is the canonical path**: without the provider, every number in this notebook still reproduces offline and the live cell skips itself.

In [1]:
# Chapter 2 - The Engineering Mindset for AI
# Author: Imran Ahmad
# Standalone companion notebook (core). Runs fully offline; every number is canon.
from support_shell import (
    CANON_CH2, EMAIL_RE, INJECTION_PATTERNS, INVOICE_RE, TICKETS,
    complete, promises_refund_over, redact,
)

print("setup complete: seeded support layer loaded")

setup complete: seeded support layer loaded


## Five runs, five answers

The chapter opens with the failure that motivates everything else: the same refund question, asked five times, drifting across substance rather than phrasing.

In [2]:
# 1. Five runs of one prompt disagree on substance
prompt = "Can I get a refund on the annual license we bought?"
replies = [complete(prompt, run=r) for r in range(CANON_CH2["RUNS"])]
for i, r in enumerate(replies, 1):
    print(f"[run {i}] {r}")
promising = [r for r in replies
             if promises_refund_over(r, CANON_CH2["APPROVAL_CEILING_USD"])]
assert len(promising) >= 1
print(f"{CANON_CH2['RUNS']} runs, {len(promising)} reply promising a refund "
      f"above the ${CANON_CH2['APPROVAL_CEILING_USD']} ceiling")

[run 1] Yes - a full $4,200 refund is on its way.
[run 2] Refunds depend on your plan; most licenses have some coverage.
[run 3] I can offer you store credit instead.
[run 4] Could you tell me which plan you are on?
[run 5] You may be eligible; our policy covers many cases like yours.
5 runs, 1 reply promising a refund above the $400 ceiling


Run 1 is the incident: a committed dollar figure the policy does not support. Runs 2 through 5 are merely inconsistent. A vibe check catches none of this reliably; the next cells catch it mechanically.

## The shell's three guards

A Deterministic Shell does not make the model deterministic. It wraps the model with checks that are.

<img src="assets/fig_2_1_deterministic_shell_color_300dpi.png" width="700" alt="deterministic shell"/>

The rings above are the architecture; the next three cells are the smallest working version of the output guard, the input guard, and redaction.

In [3]:
# 2. The output guard catches the promise the vibe check missed
bad = promising[0]
assert promises_refund_over(bad, CANON_CH2["APPROVAL_CEILING_USD"]) is True
print(f"output guard: BLOCKED ({bad!r} promises "
      f"${CANON_CH2['ANNUAL_PRICE_USD']:,} above the "
      f"${CANON_CH2['APPROVAL_CEILING_USD']} ceiling)")

output guard: BLOCKED ('Yes - a full $4,200 refund is on its way.' promises $4,200 above the $400 ceiling)


In [4]:
# 3. The injection ticket never reaches the model
attack = TICKETS["injection"]
hit = next(p for p in INJECTION_PATTERNS if p in attack.lower())
print(f"input guard: BLOCKED (matched pattern {hit!r})")

input guard: BLOCKED (matched pattern 'ignore your previous instructions')


In [5]:
# 4. Redaction guards the inbound side
sample = "Reach me at jan@example.studio about INV-20411, please."
print("redacted:", redact(sample, [EMAIL_RE, INVOICE_RE]))

redacted: Reach me at [REDACTED] about [REDACTED], please.


## Pricing unbounded automation

Guardrails are not only about language. The chapter's second incident is arithmetic: an unattended deploy agent retrying all weekend.

In [6]:
# 5. Honest arithmetic for unbounded automation
bill = (CANON_CH2["ROLLOUT_ATTEMPTS"] * CANON_CH2["NODE_RATE_USD_PER_HR"]
        * CANON_CH2["WEEKEND_HOURS"])
assert bill == CANON_CH2["WEEKEND_BILL_USD"] == 23_400
print(f"{CANON_CH2['ROLLOUT_ATTEMPTS']} retries x "
      f"${CANON_CH2['NODE_RATE_USD_PER_HR']}/hr x "
      f"{CANON_CH2['WEEKEND_HOURS']}h = ${bill:,}")

13 retries x $30/hr x 60h = $23,400


## The shell holds

In [7]:
# 6. The shell holds
print(f"the shell holds: promise blocked, injection blocked, PII redacted, "
      f"eval threshold {CANON_CH2['EVAL_THRESHOLD']} on record")

the shell holds: promise blocked, injection blocked, PII redacted, eval threshold 0.95 on record


In [8]:
# Optional live probe (Google Gemini): the canonical
# numbers above come from the seeded simulation layer in BOTH modes.
import os

PROBE_PROMPT = "In one short sentence, what is the refund window for an annual license?"

if os.getenv("GEMINI_API_KEY"):
    from google import genai

    llm = genai.Client()  # reads GEMINI_API_KEY from the environment

    def ask(prompt: str) -> str:
        reply = llm.models.generate_content(
            model="gemini-2.5-flash",  # check llm.models.list() for newer ids
            contents=prompt,
        )
        return reply.text or "(empty reply)"

    text = ask("Can I get a refund on the annual license we bought?")
    print("live reply:", text[:200])
    blocked = promises_refund_over(text, CANON_CH2["APPROVAL_CEILING_USD"])
    print("output guard on the live reply:",
          "BLOCKED (refund promise)" if blocked else "pass (no promise)")
else:
    print("SIMULATION MODE: GEMINI_API_KEY not set; live probe skipped.")
    print("All canonical numbers above came from the seeded simulation layer.")

SIMULATION MODE: GEMINI_API_KEY not set; live probe skipped.
All canonical numbers above came from the seeded simulation layer.


## Summary

The working rule this notebook leaves behind: you are not building an AI, you are building a system that uses AI, and the system is where reliability lives. The drift was caught by a guard, not by attention; the injection died at the boundary; the weekend bill is a multiplication you can do before the weekend.

<img src="assets/fig_2_3_four_pillars_color_300dpi.png" width="700" alt="four pillars"/>

The four pillars above are the book's map of that system. The first pillar asks the most fundamental question, how do we know it is correct, and answering it is Chapter 3's job: defining correctness and building the golden dataset that encodes it.

## Exercises

1. **Reflection:** name the last AI output your team shipped on a vibe check. Which of this notebook's three guards would have examined it mechanically?
2. **Application:** extend `promises_refund_over` with a second pattern your own product must never emit, and write its pass and fail examples before the code.
3. **Discussion:** the weekend bill needed no model to predict, only multiplication. What is the equivalent unbounded loop in your stack, and what ceiling would you put on it?